# 🛒 التنبؤ برحيل العملاء ونشر الموديل (Customer Churn Prediction + Deployment)
### مشروع B1 — مسار تعلم الآلة (Machine Learning Track) ⭐

---
## 🎯 المشكلة التجارية (Business Problem)
شركة تجارة إلكترونية بتخسر عملاء بصمت. **اكتساب عميل جديد بيكلّف 5–7 أضعاف** الحفاظ على عميل حالي.
المطلوب: نموذج يتنبأ **مين العملاء اللي قرّبوا يرحلوا (Churn)** قبل ما يرحلوا فعلاً، عشان فريق
التسويق يستهدفهم بعروض احتفاظ (Retention) — وده بيوفّر فلوس حقيقية.

**نوع المشكلة:** تصنيف ثنائي (Binary Classification) — هل العميل هيرحل (1) أو لا (0)؟

## 📦 ما الذي يثبته هذا المشروع في بورتفوليوك
- بناء **هدف بدون تسريب بيانات (No Data Leakage)** عبر نافذة زمنية
- خط أنابيب احترافي (`Pipeline` + `ColumnTransformer`)
- التعامل مع **البيانات غير المتوازنة (Imbalanced Data)**
- مقارنة موديلات وصولاً لـ **XGBoost**
- تقييم عميق (**ROC-AUC, PR-AUC, Threshold Tuning**) + **تفسير بـ SHAP**
- **نشر الموديل (Deployment)** كـ API بـ FastAPI (في مجلد `src/`)


## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

In [ ]:
# 🚀 إعداد تلقائي — Colab / Kaggle / محلي (no-op محلياً)
import os, sys, subprocess, importlib.util
REPO    = "https://github.com/ahmedelgendy11/AI-and-data-science-portfolio"
PROJECT = "ml/b1_churn_prediction"          # مسار المشروع داخل portfolio/
PKGS    = ["shap", "xgboost"]          # مكتبات المشروع (تتثبّت لو ناقصة)
for _pkg in PKGS:
    if importlib.util.find_spec(_pkg.replace("-", "_")) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", _pkg])
if not os.path.isdir("data"):                 # سحابياً: نجيب الريبو ونروح لمجلد المشروع
    _clone = REPO.rstrip("/").split("/")[-1]
    if not os.path.isdir(_clone):
        subprocess.run(["git", "clone", "-q", REPO + ".git"])
    os.chdir(os.path.join(_clone, "portfolio", PROJECT))
print("✓ جاهز —", os.getcwd())

## 🚀 إعداد التشغيل (Colab · Kaggle · محلي)
الخلية الجاية بتثبّت المكتبات الناقصة وتجيب الداتا تلقائياً على Colab/Kaggle.
**محلياً** (بالـ env بتاع المشروع) هي مجرد no-op — تقدر تتجاهلها.

## 📚 قبل ما تبدأ — محتاج تذاكر إيه
| المفهوم | المصدر | بيُستخدم في إيه |
|---|---|---|
| RFM & Customer Aggregation | McKinney — *Python for Data Analysis* (ch.10 groupby) | بناء ميزات على مستوى العميل |
| **Data Leakage & زمن المراقبة** | Géron — *Hands-On ML* (ch.2) / Huyen | أخطر غلطة في ML — لازم تفهمها كويس |
| Train/Test Split & Stratify | ISLR (ch.5) | تقييم عادل للموديل |
| Pipeline & ColumnTransformer | Géron (ch.2) | منع التسريب وتنظيم المعالجة |
| Imbalanced Data (class_weight, SMOTE) | Géron (ch.3) / Thakur | بيانات الـ churn دايماً غير متوازنة |
| مقاييس التصنيف (Precision/Recall/F1/ROC-AUC/PR-AUC) | ISLR (ch.4) / Géron | accuracy لوحدها بتكدب في البيانات غير المتوازنة |
| XGBoost | Thakur — *Approaching Almost Any ML Problem* | الموديل القياسي للبيانات الجدولية |
| SHAP (تفسير الموديل) | Molnar — *Interpretable ML* | شرح "ليه" الموديل قال كده — مطلوب في الشركات |

> 🎯 **بيُستخدم في الواقع:** البنوك (هجر العملاء)، شركات الاتصالات (Telco churn)، الاشتراكات (Netflix/Spotify)،
> أي بيزنس قائم على عملاء متكررين.


## 0️⃣ تجهيز المكتبات

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')
RANDOM_STATE = 42
print('Libraries ready ✓')

Libraries ready ✓


## 1️⃣ تحميل البيانات والاستكشاف (EDA)
البيانات على مستوى **المعاملة (transaction)** — كل صف عملية شراء. هنحوّلها لمستوى **العميل** بعد شوية.

In [2]:
df = pd.read_csv('data/ecommerce_transactions.csv')
# التواريخ بصيغ مختلطة → تحويل صريح آمن (errors='coerce' يحوّل التالف لـ NaT)
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce')
df = df.dropna(subset=['order_date']).drop_duplicates()
print('Shape after cleaning:', df.shape)
print('Date range:', df['order_date'].min().date(), '→', df['order_date'].max().date())
print('Unique customers:', df['customer_id'].nunique())
df.head()

Shape after cleaning: (14998, 10)
Date range: 2022-07-03 → 2023-12-31
Unique customers: 1600


,order_id,customer_id,order_date,product_name,category,unit_price,quantity,total_amount,country,payment_method
0,ORD113311,CUST2448,2023-10-13,Monitor,Electronics,1166.93,2,2333.86,Canada,PayPal
1,ORD112626,CUST2379,2023-03-23,Perfume,Beauty,75.84,2,151.68,Australia,Credit Card
2,ORD106393,CUST1671,2023-03-30,T-Shirt,Clothing,176.91,1,176.91,USA,PayPal
3,ORD104990,CUST1531,2023-08-21,Monitor,Electronics,422.58,1,422.58,Germany,PayPal
4,ORD112461,CUST2353,2023-01-19,Jacket,Clothing,84.40,3,253.20,Germany,PayPal


In [3]:
# نظرة على القيم المفقودة والأنواع
print(df.dtypes)
print('\nMissing values:\n', df.isna().sum())

order_id                     str
customer_id                  str
order_date        datetime64[us]
product_name                 str
category                     str
unit_price               float64
quantity                   int64
total_amount             float64
country                      str
payment_method               str
dtype: object

Missing values:
 order_id            0
customer_id         0
order_date          0
product_name        0
category            0
unit_price          0
quantity            0
total_amount        0
country           112
payment_method    112
dtype: int64


### استكشاف بصري سريع

In [4]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
df['category'].value_counts().plot(kind='bar', ax=axes[0], title='Orders by Category')
df.groupby(df['order_date'].dt.to_period('M').astype(str))['total_amount'].sum().plot(
    ax=axes[1], title='Monthly Revenue')
plt.tight_layout(); plt.show()

C:\Users\DELL\AppData\Local\Temp\claude\ipykernel_37028\1814319704.py:5: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 2️⃣ بناء الهدف بدون تسريب (Churn Label — No Leakage) ⚠️

**الفخّ الكلاسيكي:** لو عرّفنا "العميل اللي مطلبش من 90 يوم = churn"، وفي نفس الوقت استخدمنا
الـ Recency كميزة → الموديل هيغش (الـ Recency بتحدّد الإجابة مباشرة). ده **تسريب بيانات (Data Leakage)**.

**الحل الصح — نقسم الزمن:**
- `snapshot` = آخر تاريخ في البيانات
- `cutoff` = snapshot − 90 يوم
- **الميزات** تُحسب من المعاملات **قبل** الـ cutoff فقط (التاريخ المعروف)
- **الهدف (churn)** = هل العميل اختفى في آخر 90 يوم (من cutoff لـ snapshot)؟

كده الميزات بتتحسب من "الماضي" والهدف من "المستقبل" — زي الواقع بالظبط.


In [5]:
snapshot = df['order_date'].max()
cutoff = snapshot - pd.Timedelta(days=90)
print('Snapshot:', snapshot.date(), '| Cutoff:', cutoff.date())

hist = df[df['order_date'] < cutoff].copy()      # الماضي → للميزات
future = df[df['order_date'] >= cutoff].copy()    # المستقبل → للهدف

active_future = set(future['customer_id'].unique())
print('Customers in history:', hist['customer_id'].nunique())

Snapshot: 2023-12-31 | Cutoff: 2023-10-02
Customers in history: 1588


### هندسة الميزات على مستوى العميل (من الماضي فقط)

In [6]:
def safe_mode(x):
    m = x.mode()
    return m.iat[0] if not m.empty else 'Unknown'   # بعض العملاء عندهم القيمة مفقودة

g = hist.groupby('customer_id')
features = pd.DataFrame({
    'recency':        (cutoff - g['order_date'].max()).dt.days,   # كام يوم من آخر طلب
    'tenure':         (cutoff - g['order_date'].min()).dt.days,   # قِدَم العميل
    'frequency':      g['order_id'].nunique(),                    # عدد الطلبات
    'monetary':       g['total_amount'].sum(),                    # إجمالي الإنفاق
    'avg_order_value':g['total_amount'].mean(),
    'total_items':    g['quantity'].sum(),
    'n_categories':   g['category'].nunique(),
    'main_country':   g['country'].agg(safe_mode),
    'main_payment':   g['payment_method'].agg(safe_mode),
}).reset_index()

# الهدف: 1 لو العميل لم يظهر في نافذة المستقبل
features['churn'] = (~features['customer_id'].isin(active_future)).astype(int)
print('Churn rate: {:.1%}'.format(features['churn'].mean()))
features.head()

Churn rate: 78.9%


,customer_id,recency,tenure,frequency,monetary,avg_order_value,total_items,n_categories,main_country,main_payment,churn
0,CUST1000,288,340,2,1682.48,841.240000,3,2,France,Debit Card,1
1,CUST1001,111,132,3,267.29,89.096667,6,3,USA,PayPal,1
2,CUST1002,20,213,22,3990.96,181.407273,36,5,USA,Credit Card,1
3,CUST1003,202,336,9,763.70,84.855556,14,4,USA,PayPal,1
4,CUST1004,245,323,2,471.19,235.595000,2,2,UK,Debit Card,1


> 📌 **لاحظ:** نسبة الـ churn مش 50/50 → بيانات **غير متوازنة**. ده هيأثر على اختيار المقياس والموديل.

## 3️⃣ التقسيم والمعالجة (Train/Test + Pipeline)
بنستخدم `stratify` عشان نحافظ على نسبة الـ churn في التدريب والاختبار،
و`ColumnTransformer` عشان نعالج الأرقام والفئات كلٌ بطريقته **داخل** الـ Pipeline (يمنع التسريب).

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

X = features.drop(columns=['customer_id', 'churn'])
y = features['churn']

num_cols = ['recency','tenure','frequency','monetary','avg_order_value','total_items','n_categories']
cat_cols = ['main_country','main_payment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])
print('Train:', X_train.shape, '| Test:', X_test.shape)

Train: (1191, 9) | Test: (397, 9)


## 4️⃣ مقارنة الموديلات مع Cross-Validation
نقارن 3 موديلات: Logistic Regression (خط أساس) → Random Forest → **XGBoost**.
المقياس **ROC-AUC** (مناسب للبيانات غير المتوازنة، ومش بيتأثر بالـ threshold).
لاحظ `class_weight='balanced'` / `scale_pos_weight` للتعامل مع عدم التوازن.

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
models = {
    'LogReg': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'RandomForest': RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                           random_state=RANDOM_STATE),
    'XGBoost': XGBClassifier(n_estimators=400, learning_rate=0.05, max_depth=4,
                             scale_pos_weight=pos_weight, eval_metric='logloss',
                             random_state=RANDOM_STATE),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
for name, model in models.items():
    pipe = Pipeline([('prep', preprocess), ('clf', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
    print(f'{name:14} ROC-AUC = {scores.mean():.3f} ± {scores.std():.3f}')

LogReg         ROC-AUC = 0.924 ± 0.025


RandomForest   ROC-AUC = 0.909 ± 0.026


XGBoost        ROC-AUC = 0.904 ± 0.024


## 5️⃣ تدريب أفضل موديل وتقييمه بعمق
نختار XGBoost، ندرّبه على كامل التدريب، ونقيّمه على الاختبار بـ:
Confusion Matrix · Classification Report · ROC Curve · Precision-Recall Curve.

In [9]:
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             RocCurveDisplay, PrecisionRecallDisplay)

best = Pipeline([('prep', preprocess), ('clf', models['XGBoost'])])
best.fit(X_train, y_train)
proba = best.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print('ROC-AUC :', round(roc_auc_score(y_test, proba), 3))
print('PR-AUC  :', round(average_precision_score(y_test, proba), 3))
print('\n', classification_report(y_test, pred))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Confusion Matrix'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('Actual')
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[1]); axes[1].set_title('ROC Curve')
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[2]); axes[2].set_title('PR Curve')
plt.tight_layout(); plt.show()

ROC-AUC : 0.907
PR-AUC  : 0.977

               precision    recall  f1-score   support

           0       0.59      0.81      0.68        84
           1       0.94      0.85      0.89       313

    accuracy                           0.84       397
   macro avg       0.77      0.83      0.79       397
weighted avg       0.87      0.84      0.85       397



C:\Users\DELL\AppData\Local\Temp\claude\ipykernel_37028\531582220.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 6️⃣ ضبط عتبة القرار (Threshold Tuning) 🎯
الـ 0.5 مش دايماً الأفضل. في الـ churn، **فقدان عميل (False Negative) أغلى** من إزعاج عميل بعرض (False Positive).
نختار العتبة اللي بتعظّم الـ F1 (أو حسب تكلفة البيزنس).

In [10]:
from sklearn.metrics import precision_recall_curve, f1_score

prec, rec, thr = precision_recall_curve(y_test, proba)
f1s = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)
best_thr = thr[np.argmax(f1s)]
print(f'Best threshold = {best_thr:.2f} | F1 @0.5 = {f1_score(y_test, pred):.3f}'
      f' | F1 @best = {f1_score(y_test, (proba>=best_thr).astype(int)):.3f}')

Best threshold = 0.39 | F1 @0.5 = 0.894 | F1 @best = 0.905


## 7️⃣ تفسير الموديل بـ SHAP (Interpretability) 🔍
SHAP بيقولنا **كل ميزة ساهمت بكام** في القرار — ده اللي بيقنع المدير/العميل بالموديل.

In [11]:
import shap
# نحوّل البيانات بالـ preprocessor ثم نفسّر موديل XGBoost
Xt_test = best.named_steps['prep'].transform(X_test)
feat_names = best.named_steps['prep'].get_feature_names_out()
explainer = shap.TreeExplainer(best.named_steps['clf'])
shap_values = explainer.shap_values(Xt_test)
shap.summary_plot(shap_values, Xt_test, feature_names=feat_names, show=True)

C:\Users\DELL\miniconda3\envs\dsportfolio\Lib\site-packages\shap\plots\_beeswarm.py:1152: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8️⃣ حفظ الموديل للنشر (Model Serialization)
نحفظ الـ Pipeline كامل (المعالجة + الموديل) بـ joblib عشان نستخدمه في الـ API.

In [12]:
import joblib, os
os.makedirs('src', exist_ok=True)
joblib.dump(best, 'src/churn_model.joblib')
print('Saved → src/churn_model.joblib')

Saved → src/churn_model.joblib


## 9️⃣ النشر (Deployment) — FastAPI
الكود الكامل للـ API في `src/app.py`. لتشغيله:
```bash
cd src
uvicorn app:app --reload
# ثم افتح http://127.0.0.1:8000/docs لتجربة الـ endpoint
```
ده بيحوّل الموديل من نوتبوك لـ **خدمة حقيقية** ممكن أي تطبيق يستدعيها — ده الجزء اللي بيفرّق في البورتفوليو.

## 🔟 الخلاصة والتوصيات التجارية (Conclusion)
- **النتيجة:** XGBoost حقّق أعلى ROC-AUC، ويقدر يحدّد العملاء المعرّضين للرحيل بدقة كويسة.
- **أهم العوامل (من SHAP):** `recency` و `frequency` و `monetary` — العميل اللي بقاله مدة وقلّت طلباته أخطر.
- **التوصية:** فريق الاحتفاظ يستهدف العملاء فوق العتبة المختارة بعروض مخصّصة → توفير تكلفة اكتساب.
- **الخطوات القادمة:** إضافة ميزات سلوكية (زيارات الموقع)، تتبّع الأداء بعد النشر (Data Drift)، وإعادة التدريب دورياً.

> ✅ **اللي اتعلمته هنا:** تجنّب التسريب، Pipeline، عدم التوازن، XGBoost، تقييم عميق، SHAP، والنشر.
